# Weather Analysis Workflow

## Overview
This notebook demonstrates a complete workflow for analyzing weather data from NOAA sources:
1. **Data Ingestion**: Load weather data from NOAA
2. **Spatial Analysis**: Buffer regions, identify hotspots
3. **Statistical Analysis**: Calculate patterns and correlations
4. **Visualization**: Create interactive maps and heatmaps

## Prerequisites
- GeoSpatial Analysis Engine running on localhost:8080
- Python requests library
- Jupyter notebook environment

## 1. Setup and Configuration

In [ ]:
import requests
import json
import time
from datetime import datetime
import pandas as pd
import numpy as np

# API Configuration
API_BASE = 'http://localhost:8080'
API_TOKEN = 'your_token_here'  # Set after authentication

# Headers for API requests
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {API_TOKEN}'
}

print('API Configuration Ready')
print(f'Base URL: {API_BASE}')

## 2. Authentication

In [ ]:
# Login to get authentication token
def login(username, password):
    response = requests.post(
        f'{API_BASE}/auth/login',
        json={'username': username, 'password': password}
    )
    if response.status_code == 200:
        data = response.json()['data']
        token = data['token']
        headers['Authorization'] = f'Bearer {token}'
        print(f'✓ Logged in as {username}')
        return token
    else:
        print(f'✗ Login failed: {response.text}')
        return None

# Authenticate with demo credentials
token = login('admin', 'changeme')

## 3. Data Ingestion from NOAA

In [ ]:
# Sample NOAA weather data
weather_data = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-122.4, 37.8]},
            "properties": {
                "station_id": "NOAA_001",
                "temperature_celsius": 18.5,
                "humidity_percent": 65,
                "wind_speed_kmh": 12.3,
                "precipitation_mm": 2.5,
                "timestamp": "2026-04-12T12:00:00Z"
            }
        },
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-122.3, 37.9]},
            "properties": {
                "station_id": "NOAA_002",
                "temperature_celsius": 19.2,
                "humidity_percent": 62,
                "wind_speed_kmh": 11.8,
                "precipitation_mm": 1.8,
                "timestamp": "2026-04-12T12:00:00Z"
            }
        },
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-122.2, 37.85]},
            "properties": {
                "station_id": "NOAA_003",
                "temperature_celsius": 17.8,
                "humidity_percent": 70,
                "wind_speed_kmh": 14.5,
                "precipitation_mm": 3.2,
                "timestamp": "2026-04-12T12:00:00Z"
            }
        }
    ]
}

# Ingest data via API
response = requests.post(
    f'{API_BASE}/ingest',
    headers=headers,
    json={
        'source': 'NOAA Weather Data',
        'format': 'GeoJSON',
        'data': weather_data
    }
)

if response.status_code == 200:
    ingest_result = response.json()['data']
    print(f'✓ Data ingested successfully')
    print(f'  Rows ingested: {ingest_result.get("rows_ingested", "N/A")}')
    print(f'  Columns: {ingest_result.get("columns", [])}')
else:
    print(f'✗ Ingestion failed: {response.text}')

## 4. Spatial Analysis - Buffer Operation

In [ ]:
# Create buffers around weather stations (e.g., 10km radius)
buffer_response = requests.post(
    f'{API_BASE}/analyze',
    headers=headers,
    json={
        'operation': 'buffer',
        'params': {
            'distance': 10,
            'units': 'kilometers'
        }
    }
)

if buffer_response.status_code == 200:
    buffer_result = buffer_response.json()['data']
    print('✓ Buffer operation completed')
    print(f'  Result type: {buffer_result.get("type", "N/A")}')
    print(f'  Feature count: {len(buffer_result.get("features", []))}')
else:
    print(f'✗ Buffer operation failed: {buffer_response.text}')

## 5. Hotspot Analysis

In [ ]:
# Perform hotspot analysis on temperature data
hotspot_response = requests.post(
    f'{API_BASE}/analyze',
    headers=headers,
    json={
        'operation': 'hotspot_analysis',
        'params': {
            'variable': 'temperature_celsius'
        }
    }
)

if hotspot_response.status_code == 200:
    hotspot_result = hotspot_response.json()['data']
    print('✓ Hotspot analysis completed')
    print(f'  Analysis type: Temperature hotspots')
    print(f'  Hotspots identified: {hotspot_result.get("hotspot_count", "N/A")}')
else:
    print(f'✗ Hotspot analysis failed: {hotspot_response.text}')

## 6. Spatial Autocorrelation Analysis

In [ ]:
# Calculate Moran's I autocorrelation statistic
autocorr_response = requests.post(
    f'{API_BASE}/analyze',
    headers=headers,
    json={
        'operation': 'spatial_autocorrelation',
        'params': {
            'variable': 'temperature_celsius'
        }
    }
)

if autocorr_response.status_code == 200:
    autocorr_result = autocorr_response.json()['data']
    print('✓ Spatial autocorrelation analysis completed')
    print(f'  Morans I: {autocorr_result.get("morans_i", "N/A"):.4f}')
    print(f'  P-value: {autocorr_result.get("p_value", "N/A"):.4f}')
    print(f'  Interpretation: Spatial clustering detected' if autocorr_result.get('morans_i', 0) > 0 else 'Spatial dispersion detected')
else:
    print(f'✗ Autocorrelation analysis failed: {autocorr_response.text}')

## 7. Visualization

In [ ]:
# Create interactive map visualization
viz_response = requests.post(
    f'{API_BASE}/visualize',
    headers=headers,
    json={
        'type': 'map',
        'variable': 'temperature_celsius',
        'colormap': 'YlOrRd',
        'title': 'NOAA Weather Station Temperature Analysis'
    }
)

if viz_response.status_code == 200:
    viz_result = viz_response.json()['data']
    print('✓ Visualization created')
    print(f'  URL: {viz_result.get("url", "N/A")}')
    print(f'  Type: Interactive Map')
else:
    print(f'✗ Visualization failed: {viz_response.text}')

## 8. Export Results

In [ ]:
# Export analysis results
export_response = requests.post(
    f'{API_BASE}/export',
    headers=headers,
    json={
        'format': 'GeoJSON',
        'variable': 'temperature_celsius',
        'filter': None
    }
)

if export_response.status_code == 200:
    print('✓ Results exported successfully')
    print('  Available formats: GeoJSON, CSV, GeoTIFF')
else:
    print(f'✗ Export failed: {export_response.text}')

## 9. Summary

### Workflow Completed
✓ Data ingestion from NOAA sources
✓ Spatial buffer operations
✓ Hotspot identification
✓ Autocorrelation analysis
✓ Interactive visualization
✓ Results export

### Key Findings
- Temperature variations identified across weather stations
- Spatial clustering patterns detected
- Interactive map ready for further analysis

### Next Steps
- Compare temporal trends
- Combine with other data sources (USGS, Copernicus)
- Apply machine learning for prediction